## 1. Load Dataset

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
df = pd.read_csv('data.csv')
df.head()

,Unnamed: 0,Name,Rating,Branches,Highly_rated,Critically_rated,Reviews,Salaries,Interviews,Jobs,Benefits,Logo_URL
0,0,TCS,3.8,4.0,"Job Security, Work Life Balance, Company Culture","Promotions / Appraisal, Salary & Benefits",67k,737.1k,5.6k,278,11.3k,https://static.ambitionbox.com/alpha/company/p...
1,1,Accenture,4.1,2.0,"Company Culture, Job Security, Skill Developme...",NaN,42.9k,513.8k,3.9k,4.1k,7k,https://static.ambitionbox.com/assets/v2/image...
2,2,Cognizant,3.9,2.0,Skill Development / Learning,NaN,38.6k,497.3k,3.3k,480,5.8k,https://static.ambitionbox.com/alpha/company/p...
3,3,Wipro,3.8,3.0,Job Security,NaN,35.6k,371.3k,3.3k,340,4.9k,https://static.ambitionbox.com/assets/v2/image...
4,4,ICICI Bank,4.0,2.0,"Job Security, Skill Development / Learning, Co...",NaN,31k,136.4k,1.7k,216,3.7k,https://static.ambitionbox.com/assets/v2/image...


In [3]:
df.shape

(10000, 12)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        10000 non-null  int64  
 1   Name              10000 non-null  object 
 2   Rating            10000 non-null  float64
 3   Branches          2729 non-null   float64
 4   Highly_rated      8029 non-null   object 
 5   Critically_rated  237 non-null    object 
 6   Reviews           10000 non-null  object 
 7   Salaries          10000 non-null  object 
 8   Interviews        10000 non-null  object 
 9   Jobs              10000 non-null  object 
 10  Benefits          10000 non-null  object 
 11  Logo_URL          10000 non-null  object 
dtypes: float64(2), int64(1), object(9)
memory usage: 937.6+ KB


In [5]:
df.isnull().sum()

,0
Unnamed: 0,0
Name,0
Rating,0
Branches,7271
Highly_rated,1971
Critically_rated,9763
Reviews,0
Salaries,0
Interviews,0
Jobs,0


## 2. Pemeriksaan Kolom yang Tidak Digunakan

In [6]:
excluded_cols = ['Branches', 'Highly_rated', 'Critically_rated']

excluded_check = pd.DataFrame({
    'missing_count': df[excluded_cols].isnull().sum(),
    'missing_percent': df[excluded_cols].isnull().mean() * 100,
    'dtype': df[excluded_cols].dtypes.astype(str),
    'unique_count': df[excluded_cols].nunique(dropna=True)
})

excluded_check

,missing_count,missing_percent,dtype,unique_count
Branches,7271,72.71,float64,9
Highly_rated,1971,19.71,object,252
Critically_rated,9763,97.63,object,29


## 3. Konversi Data Numerik

In [7]:
def convert_k(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    if x in ['--', '-', '', 'nan']:
        return np.nan
    if 'k' in x:
        return float(x.replace('k', '')) * 1000
    return float(x)

In [8]:
input_cols = ['Reviews', 'Salaries', 'Interviews', 'Jobs', 'Benefits']
output_col = 'Rating'

for col in input_cols:
    df[col] = df[col].apply(convert_k)

missing_after_conversion = df[input_cols].isnull().sum()
missing_after_conversion

,0
Reviews,0
Salaries,0
Interviews,279
Jobs,4047
Benefits,51


## 4. Drop Row Missing Value

In [9]:
rows_before_drop = len(df)

df_model = df[input_cols + [output_col]].dropna().reset_index(drop=True)

rows_after_drop = len(df_model)
rows_removed = rows_before_drop - rows_after_drop

pd.DataFrame({
    'keterangan': ['sebelum_drop', 'setelah_drop', 'baris_dihapus'],
    'jumlah_baris': [rows_before_drop, rows_after_drop, rows_removed]
})

,keterangan,jumlah_baris
0,sebelum_drop,10000
1,setelah_drop,5867
2,baris_dihapus,4133


In [10]:
df_model.isnull().sum()

,0
Reviews,0
Salaries,0
Interviews,0
Jobs,0
Benefits,0
Rating,0


In [11]:
df_model.head()

,Reviews,Salaries,Interviews,Jobs,Benefits,Rating
0,67000.0,737100.0,5600.0,278.0,11300.0,3.8
1,42900.0,513800.0,3900.0,4100.0,7000.0,4.1
2,38600.0,497300.0,3300.0,480.0,5800.0,3.9
3,35600.0,371300.0,3300.0,340.0,4900.0,3.8
4,31000.0,136400.0,1700.0,216.0,3700.0,4.0


In [12]:
df_model.describe()

,Reviews,Salaries,Interviews,Jobs,Benefits,Rating
count,5867.000000,5867.000000,5867.000000,5867.000000,5867.000000,5867.000000
mean,517.356571,3180.664735,32.848304,32.282768,73.414863,3.822090
std,1909.768614,18029.463444,159.989042,113.937823,288.245898,0.379076
min,66.000000,5.000000,1.000000,1.000000,1.000000,1.600000
25%,99.000000,565.000000,5.000000,3.000000,12.000000,3.600000
50%,166.000000,959.000000,10.000000,9.000000,23.000000,3.900000
75%,360.000000,2000.000000,22.000000,26.000000,53.000000,4.100000
max,67000.000000,737100.000000,5600.000000,4100.000000,11300.000000,4.900000


In [13]:
X = df_model[input_cols]
y = df_model[output_col]

## 5. Fungsi Keanggotaan

In [14]:
def linier_naik(x, a, b):
    if x <= a:
        return 0.0
    if x >= b:
        return 1.0
    return (x - a) / (b - a)


def linier_turun(x, a, b):
    if x <= a:
        return 1.0
    if x >= b:
        return 0.0
    return (b - x) / (b - a)


def segitiga(x, a, b, c):
    if x <= a or x >= c:
        return 0.0
    if x == b:
        return 1.0
    if a < x < b:
        return (x - a) / (b - a)
    return (c - x) / (c - b)


def trapesium(x, a, b, c, d):
    if x <= a:
        if a == b and x == a:
            return 1.0
        return 0.0
    if x >= d:
        if c == d and x == d:
            return 1.0
        return 0.0
    if b <= x <= c:
        return 1.0
    if a < x < b:
        return (x - a) / (b - a)
    return (d - x) / (d - c)


def hitung_mf(x, params):
    if params[0] == 'linier_naik':
        return linier_naik(x, params[1], params[2])
    if params[0] == 'linier_turun':
        return linier_turun(x, params[1], params[2])
    if params[0] == 'segitiga':
        return segitiga(x, params[1], params[2], params[3])
    if params[0] == 'trapesium':
        return trapesium(x, params[1], params[2], params[3], params[4])
    raise ValueError('Jenis fungsi keanggotaan tidak dikenal')

## 6. Variabel Linguistik Input

In [15]:
batas_linguistik = pd.DataFrame({
    'variabel': ['Reviews', 'Salaries', 'Interviews', 'Jobs', 'Benefits'],
    'a': [76, 395, 3, 1, 7],
    'b': [99, 565, 5, 3, 12],
    'c': [166, 959, 10, 9, 23],
    'd': [360, 2000, 22, 26, 53],
    'e': [970, 5100, 55, 66, 136]
})

batas_linguistik

,variabel,a,b,c,d,e
0,Reviews,76,99,166,360,970
1,Salaries,395,565,959,2000,5100
2,Interviews,3,5,10,22,55
3,Jobs,1,3,9,26,66
4,Benefits,7,12,23,53,136


In [16]:
def buat_parameter_input(a, b, c, d, e):
    return {
        'sangat_rendah': ('linier_turun', a, b),
        'rendah': ('segitiga', a, b, c),
        'sedang': ('segitiga', b, c, d),
        'tinggi': ('segitiga', c, d, e),
        'sangat_tinggi': ('linier_naik', d, e)
    }


input_params = {}

for _, row in batas_linguistik.iterrows():
    input_params[row['variabel']] = buat_parameter_input(row['a'], row['b'], row['c'], row['d'], row['e'])

input_params

{'Reviews': {'sangat_rendah': ('linier_turun', 76, 99),
  'rendah': ('segitiga', 76, 99, 166),
  'sedang': ('segitiga', 99, 166, 360),
  'tinggi': ('segitiga', 166, 360, 970),
  'sangat_tinggi': ('linier_naik', 360, 970)},
 'Salaries': {'sangat_rendah': ('linier_turun', 395, 565),
  'rendah': ('segitiga', 395, 565, 959),
  'sedang': ('segitiga', 565, 959, 2000),
  'tinggi': ('segitiga', 959, 2000, 5100),
  'sangat_tinggi': ('linier_naik', 2000, 5100)},
 'Interviews': {'sangat_rendah': ('linier_turun', 3, 5),
  'rendah': ('segitiga', 3, 5, 10),
  'sedang': ('segitiga', 5, 10, 22),
  'tinggi': ('segitiga', 10, 22, 55),
  'sangat_tinggi': ('linier_naik', 22, 55)},
 'Jobs': {'sangat_rendah': ('linier_turun', 1, 3),
  'rendah': ('segitiga', 1, 3, 9),
  'sedang': ('segitiga', 3, 9, 26),
  'tinggi': ('segitiga', 9, 26, 66),
  'sangat_tinggi': ('linier_naik', 26, 66)},
 'Benefits': {'sangat_rendah': ('linier_turun', 7, 12),
  'rendah': ('segitiga', 7, 12, 23),
  'sedang': ('segitiga', 12, 23, 

In [17]:
def nilai_keanggotaan_input(x, params):
    hasil = {}
    for label, p in params.items():
        hasil[label] = hitung_mf(x, p)
    return hasil


def fuzzifikasi(row):
    hasil = {}
    for col in input_cols:
        hasil[col] = nilai_keanggotaan_input(row[col], input_params[col])
    return hasil


fuzzifikasi(X.iloc[0])

{'Reviews': {'sangat_rendah': 0.0,
  'rendah': 0.0,
  'sedang': 0.0,
  'tinggi': 0.0,
  'sangat_tinggi': 1.0},
 'Salaries': {'sangat_rendah': 0.0,
  'rendah': 0.0,
  'sedang': 0.0,
  'tinggi': 0.0,
  'sangat_tinggi': 1.0},
 'Interviews': {'sangat_rendah': 0.0,
  'rendah': 0.0,
  'sedang': 0.0,
  'tinggi': 0.0,
  'sangat_tinggi': 1.0},
 'Jobs': {'sangat_rendah': 0.0,
  'rendah': 0.0,
  'sedang': 0.0,
  'tinggi': 0.0,
  'sangat_tinggi': 1.0},
 'Benefits': {'sangat_rendah': 0.0,
  'rendah': 0.0,
  'sedang': 0.0,
  'tinggi': 0.0,
  'sangat_tinggi': 1.0}}

## 7. Variabel Linguistik Output Rating

In [18]:
rating_domain = np.linspace(1, 5, 401)

rating_params = {
    'sangat_buruk': ('trapesium', 1, 1, 1, 2),
    'buruk': ('segitiga', 1, 2, 3),
    'cukup': ('segitiga', 2, 3, 4),
    'baik': ('segitiga', 3, 4, 5),
    'sangat_baik': ('trapesium', 4, 5, 5, 5)
}

rating_mf = {}

for label, params in rating_params.items():
    rating_mf[label] = np.array([hitung_mf(x, params) for x in rating_domain])

list(rating_mf.keys())

['sangat_buruk', 'buruk', 'cukup', 'baik', 'sangat_baik']

## 8. Rule Base

In [19]:
rules = []

for var in input_cols:
    rules.append(({var: 'sangat_rendah'}, 'cukup'))
    rules.append(({var: 'rendah'}, 'baik'))
    rules.append(({var: 'sedang'}, 'baik'))
    rules.append(({var: 'tinggi'}, 'baik'))
    rules.append(({var: 'sangat_tinggi'}, 'baik'))

rules.extend([
    ({'Reviews': 'sangat_tinggi', 'Salaries': 'sangat_tinggi', 'Benefits': 'sangat_tinggi'}, 'sangat_baik'),
    ({'Reviews': 'sangat_tinggi', 'Benefits': 'sangat_tinggi'}, 'sangat_baik'),
    ({'Reviews': 'sangat_tinggi', 'Jobs': 'sangat_tinggi', 'Benefits': 'sangat_tinggi'}, 'sangat_baik'),
    ({'Salaries': 'sangat_tinggi', 'Interviews': 'sangat_tinggi', 'Jobs': 'sangat_tinggi'}, 'sangat_baik'),
    ({'Reviews': 'sangat_rendah', 'Salaries': 'sangat_rendah', 'Benefits': 'sangat_rendah', 'Interviews': 'sangat_rendah', 'Jobs': 'sangat_rendah'}, 'sangat_buruk'),
    ({'Reviews': 'sangat_rendah', 'Salaries': 'sangat_rendah', 'Benefits': 'sangat_rendah'}, 'buruk'),
    ({'Reviews': 'sangat_rendah', 'Jobs': 'sangat_rendah', 'Benefits': 'sangat_rendah'}, 'buruk'),
    ({'Salaries': 'sangat_rendah', 'Interviews': 'sangat_rendah', 'Jobs': 'sangat_rendah'}, 'buruk')
])

len(rules)

33

In [20]:
rule_table = pd.DataFrame([
    {'No': i + 1, 'IF': kondisi, 'THEN': output}
    for i, (kondisi, output) in enumerate(rules)
])

rule_table

,No,IF,THEN
0,1,{'Reviews': 'sangat_rendah'},cukup
1,2,{'Reviews': 'rendah'},baik
2,3,{'Reviews': 'sedang'},baik
3,4,{'Reviews': 'tinggi'},baik
4,5,{'Reviews': 'sangat_tinggi'},baik
5,6,{'Salaries': 'sangat_rendah'},cukup
6,7,{'Salaries': 'rendah'},baik
7,8,{'Salaries': 'sedang'},baik
8,9,{'Salaries': 'tinggi'},baik
9,10,{'Salaries': 'sangat_tinggi'},baik


In [21]:
rule_table['THEN'].value_counts()

,count
THEN,
baik,20
cukup,5
sangat_baik,4
buruk,3
sangat_buruk,1


## 9. Inferensi Mamdani

In [22]:
def inferensi_mamdani(row):
    fuzzy = fuzzifikasi(row)
    agregasi = np.zeros_like(rating_domain)

    for kondisi, output in rules:
        alpha = min(fuzzy[var][label] for var, label in kondisi.items())
        implikasi = np.minimum(alpha, rating_mf[output])
        agregasi = np.maximum(agregasi, implikasi)

    if agregasi.sum() == 0:
        return 3.8

    return np.sum(rating_domain * agregasi) / np.sum(agregasi)

## 10. Inferensi Sugeno

In [23]:
sugeno_const = {
    'sangat_buruk': 1,
    'buruk': 2,
    'cukup': 3,
    'baik': 4,
    'sangat_baik': 5
}


def inferensi_sugeno(row):
    fuzzy = fuzzifikasi(row)
    pembilang = 0.0
    penyebut = 0.0

    for kondisi, output in rules:
        alpha = min(fuzzy[var][label] for var, label in kondisi.items())
        pembilang += alpha * sugeno_const[output]
        penyebut += alpha

    if penyebut == 0:
        return 3.8

    return pembilang / penyebut

## 11. Prediksi

In [24]:
y_pred_mamdani = X.apply(inferensi_mamdani, axis=1)
y_pred_sugeno = X.apply(inferensi_sugeno, axis=1)

hasil = df_model.copy()
hasil['Prediksi_Mamdani'] = y_pred_mamdani
hasil['Prediksi_Sugeno'] = y_pred_sugeno
hasil['Error_Mamdani'] = abs(hasil['Rating'] - hasil['Prediksi_Mamdani'])
hasil['Error_Sugeno'] = abs(hasil['Rating'] - hasil['Prediksi_Sugeno'])

hasil.head()

,Reviews,Salaries,Interviews,Jobs,Benefits,Rating,Prediksi_Mamdani,Prediksi_Sugeno,Error_Mamdani,Error_Sugeno
0,67000.0,737100.0,5600.0,278.0,11300.0,3.8,4.17,4.444444,0.37,0.644444
1,42900.0,513800.0,3900.0,4100.0,7000.0,4.1,4.17,4.444444,0.07,0.344444
2,38600.0,497300.0,3300.0,480.0,5800.0,3.9,4.17,4.444444,0.27,0.544444
3,35600.0,371300.0,3300.0,340.0,4900.0,3.8,4.17,4.444444,0.37,0.644444
4,31000.0,136400.0,1700.0,216.0,3700.0,4.0,4.17,4.444444,0.17,0.444444


## 12. Evaluasi

In [25]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


hasil_evaluasi = pd.DataFrame({
    'Metode': ['Mamdani', 'Sugeno'],
    'MAE': [mae(y, y_pred_mamdani), mae(y, y_pred_sugeno)],
    'MSE': [mse(y, y_pred_mamdani), mse(y, y_pred_sugeno)]
})

hasil_evaluasi

,Metode,MAE,MSE
0,Mamdani,0.404031,0.272787
1,Sugeno,0.382993,0.249099


In [26]:
hasil[['Rating', 'Prediksi_Mamdani', 'Prediksi_Sugeno', 'Error_Mamdani', 'Error_Sugeno']].head(20)

,Rating,Prediksi_Mamdani,Prediksi_Sugeno,Error_Mamdani,Error_Sugeno
0,3.8,4.17,4.444444,0.37,0.644444
1,4.1,4.17,4.444444,0.07,0.344444
2,3.9,4.17,4.444444,0.27,0.544444
3,3.8,4.17,4.444444,0.37,0.644444
4,4.0,4.17,4.444444,0.17,0.444444
5,3.9,4.17,4.444444,0.27,0.544444
6,3.9,4.17,4.444444,0.27,0.544444
7,3.8,4.17,4.444444,0.37,0.644444
8,3.7,4.17,4.444444,0.47,0.744444
9,3.7,4.17,4.444444,0.47,0.744444


In [27]:
prediction_summary = pd.DataFrame({
    'Rating_Aktual': y.describe(),
    'Prediksi_Mamdani': y_pred_mamdani.describe(),
    'Prediksi_Sugeno': y_pred_sugeno.describe()
})

prediction_summary

,Rating_Aktual,Prediksi_Mamdani,Prediksi_Sugeno
count,5867.000000,5867.000000,5867.000000
mean,3.822090,3.722696,3.851467
std,0.379076,0.358786,0.338068
min,1.600000,2.330000,2.444444
25%,3.600000,3.499061,3.700000
50%,3.900000,3.764851,3.988235
75%,4.100000,4.000000,4.000000
max,4.900000,4.170000,4.444444


## 13. Validasi Distribusi Label Dominan Input

In [28]:
dominant_labels = {}

for col in input_cols:
    labels = []
    for value in X[col]:
        membership = nilai_keanggotaan_input(value, input_params[col])
        labels.append(max(membership, key=membership.get))
    dominant_labels[col] = pd.Series(labels).value_counts()

dominant_table = pd.DataFrame(dominant_labels).fillna(0).astype(int)
dominant_table

,Reviews,Salaries,Interviews,Jobs,Benefits
rendah,1297,1281,1101,1223,1383
sangat_rendah,1081,1053,1278,1232,966
sangat_tinggi,833,865,887,845,859
sedang,1542,1516,1640,1459,1593
tinggi,1114,1152,961,1108,1066


## 14. Validasi Ketentuan Tugas

In [29]:
hasil.to_csv('hasil_prediksi_fuzzy_final_fix.csv', index=False)
hasil_evaluasi.to_csv('evaluasi_fuzzy_final_fix.csv', index=False)

Path('hasil_prediksi_fuzzy_final_fix.csv').exists(), Path('evaluasi_fuzzy_final_fix.csv').exists()

(True, True)